# Can the revised starting weights support learning and remembering?

**Target:** train a fresh model on task 3, then task 0, then task 9. Check whether it learns each task and keeps the earlier task scores.

Initialization means choosing the model's numbers before its first lesson. This run uses the new Hyperfan-in option. It also starts BatchNorm weights at one and BatchNorm biases at zero. Those BatchNorm choices are our documented implementation choices.

There is no forgetting and no old starting checkpoint. Use one fresh Colab GPU session. Keep seed 0, beta 0.01, learning rate 0.0001, and five passes through each task's training images. This is a short check of the revised recipe. One run cannot prove that it is better across seeds.

Run the cells below from top to bottom. You do not need to run notebooks 1 through 12.


## 1. Connect Drive and load the tested code

Choose an A100 GPU in Colab's runtime settings. Run this cell and allow access to Drive. It downloads the exact training-code version selected below.


In [ ]:
from pathlib import Path
import os
import subprocess
import sys
from google.colab import drive

drive.mount("/content/drive")
REPO = Path("/content/uncle-hyperfan-pilot")
CODE_VERSION = "b1797e0a93644f501508dfb856d38ad921bb6db8"
assert not any(n == "uncle" or n.startswith("uncle.") for n in sys.modules), "Start a new Colab session before loading this code again."
if not REPO.exists():
    subprocess.run(["git", "clone", "https://github.com/sumitasthana/CARK.git", str(REPO)], check=True)
else:
    assert (REPO / ".git").is_dir(), "Use a fresh Colab session. This folder is not the project."
    remote = subprocess.check_output(["git", "-C", str(REPO), "remote", "get-url", "origin"], text=True).strip().removesuffix(".git")
    assert remote == "https://github.com/sumitasthana/CARK", "Use a fresh session. This folder belongs to another project."
    assert not subprocess.check_output(["git", "-C", str(REPO), "status", "--porcelain"], text=True).strip(), "Save local code changes and use a fresh session."
subprocess.run(["git", "-C", str(REPO), "fetch", "origin"], check=True)
subprocess.run(["git", "-C", str(REPO), "checkout", "--detach", CODE_VERSION], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "--no-deps", "-e", str(REPO)], check=True)
os.chdir(REPO)
sys.path.insert(0, str(REPO))
print("Training code loaded:", CODE_VERSION)


## 2. Choose the run folder and check the settings

The model starts from new numbers. This cell does not load `before_forgetting.pt`.

Keep the experiment name below for this run. If the session stops, open this notebook in a fresh A100 session and run the same cells with the same name. Training continues after the last fully saved task. A task interrupted halfway through starts again. To make a separate new experiment, change the name first.


In [ ]:
import hashlib
import json
import math
import torch
from uncle.config import Config, dataset_defaults

EXPERIMENT_ID = "20261006_hyperfan_L3_L0_L9_seed0_01"
TASK_ORDER = ("3", "0", "9")
OUTPUT = Path("/content/drive/MyDrive/uncle/learning_initialization") / EXPERIMENT_ID
DATA = Path("/content/data/tiny-imagenet-200")
assert torch.cuda.is_available(), "Choose a GPU runtime in Colab."
gpu_name = torch.cuda.get_device_name(torch.cuda.current_device())
assert "A100" in gpu_name, f"Choose A100 for this ResNet50 run. Colab provided {gpu_name}."
config = Config(**{**dataset_defaults("tiny_imagenet", 1),
    "requests": tuple(("learn", task) for task in TASK_ORDER),
    "initialization": "hyperfan_in", "seed": 0, "beta": 0.01,
    "learning_rate": 0.0001, "epochs": 5, "batch_size": 64,
    "device": "cuda", "data_root": str(DATA)})
contract = json.loads(json.dumps({"code_version": CODE_VERSION,
    "config": vars(config), "partition_sha256": hashlib.sha256(
        (REPO / "uncle/task_partition.json").read_bytes()).hexdigest()}))
OUTPUT.mkdir(parents=True, exist_ok=True)
contract_path = OUTPUT / "settings.json"
if contract_path.exists():
    assert json.loads(contract_path.read_text()) == contract, "This folder contains a run with other code or settings. Choose a new experiment name."
else:
    assert not any(OUTPUT.iterdir()), "This folder already contains files. Choose a new experiment name."
    contract_path.write_text(json.dumps(contract, indent=2) + "\n")
print("GPU:", gpu_name)
print("Starting model: new model, with Hyperfan-in initialization")
print("Task order:", " -> ".join(TASK_ORDER))
print("Seed:", config.seed, "| beta:", config.beta, "| learning rate:", config.learning_rate)
print("Passes through each task's training images:", config.epochs)
print("Results will be saved in:", OUTPUT)


## 3. Train and save the scores

Run this cell. Downloading the images can take several minutes. Training then prints a row after each task.

Each number is the percentage of validation images answered correctly. A blank means the task has not been taught yet. Look at the same column across rows: a fall means the model is losing an earlier ability.

The saved model and scores are updated after each completed task. Do not change the settings halfway through the run.


In [ ]:
from uncle.experiments import run_experiment

rows = []
def show_table(records):
    print("Accuracy (% of validation images answered correctly)")
    print(f"{'Stage':<22}" + "".join(f"{'Task ' + t:>12}" for t in TASK_ORDER))
    for record in records:
        scores = record["after"]
        print(f"{'After learning ' + record['task']:<22}" + "".join(
            f"{scores[t]:>12.1f}" if t in scores else f"{'':>12}" for t in TASK_ORDER))

def save_table(records):
    table = {"complete": len(records) == len(TASK_ORDER),
             "code_version": CODE_VERSION, "config": vars(config),
             "rows": [{"task": r["task"], "accuracies": r["after"]} for r in records]}
    temporary = OUTPUT / "accuracy_table.json.writing"
    temporary.write_text(json.dumps(table, indent=2) + "\n")
    os.replace(temporary, OUTPUT / "accuracy_table.json")
    return table

def record_task(record):
    rows.append(record)
    save_table(rows)
    show_table(rows)

# Load earlier rows only from this experiment's own checkpoint.
# The training entry point validates its full configuration before resuming.
from uncle import checkpoint as checkpointing
checkpoint_path = OUTPUT / "checkpoint_seq1_resnet50_seed0.pt"
saved = checkpointing.load(checkpoint_path, config)
if saved is not None:
    rows.extend(saved["history"])
    print("Continuing after", len(rows), "fully saved tasks.")
    show_table(rows)
    del saved
else:
    print("Starting fresh. No saved model is being loaded.")

result = run_experiment(config=config, root=DATA, download=True,
    output=OUTPUT, checkpoint=True, resume=True, on_request=record_task,
    verbose=False, progress=True)
assert result["history"] == rows
report = save_table(result["history"])
show_table(result["history"])
print("Training finished. Results saved in:", OUTPUT)


## 4. Check the files and release the GPU

Run this cell as soon as training finishes. It checks the saved scores and model, finishes saving Drive, and ends the GPU session.

Share the three-row accuracy table. We will check whether task 3 and task 0 stay close to their scores immediately after learning. We will also check whether task 9 learns. This run changes the starting recipe, including BatchNorm and classifier settings; it cannot tell us which individual change caused a difference.


In [ ]:
from google.colab import runtime

saved_report = json.loads((OUTPUT / "accuracy_table.json").read_text())
assert saved_report == json.loads(json.dumps(report)) and saved_report["complete"]
final = checkpointing.load(checkpoint_path, config)
assert [(r["action"], r["task"]) for r in final["history"]] == list(config.requests)
assert not final["forgotten"]
assert final["previous"] == saved_report["rows"][-1]["accuracies"]
assert all(math.isfinite(score) and 0 <= score <= 100
           for row in saved_report["rows"] for score in row["accuracies"].values())
show_table(final["history"])
del final
print("Files checked. Saving Drive and ending the GPU session.", flush=True)
drive.flush_and_unmount(timeout_ms=600000)
runtime.unassign()
